# 03 — Оценка checkpoint: два отдельных протокола

A: окна вокруг известных отметок. B: полный проход видео. Обучение и подбор порога не выполняются.

## Как интерпретировать результат
Протокол A — классификация окон вокруг **известных отметок**, а не автоматический поиск события во всём видео.
Согласованный `within_room_unique_video` split: 30 train / 10 validation; точные копии не пересекаются.
Общий кабинет, люди и неизвестные сеансы могут пересекаться. Независимость камер и сеансов не подтверждена.
Validation используется для выбора checkpoint по loss, а не как независимый финальный тест.
Первый smoke-run предсказывал нарушение для всех validation-окон (accuracy 0.5, recall 1.0).
Это проверка работоспособности, не хорошее качество модели и не законченный детектор.
Исторические метрики и provenance сохранены в `experiments/r3d18_smoke_result.md` и прежних `runs/`.
Протокол B использует только видео для предсказаний; разметка применяется после прохода для оценки.


In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/overfitted_underqualified").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите из корня репозитория или notebooks/")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import os
import time
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
from IPython.display import FileLink, display
from dotenv import dotenv_values

from overfitted_underqualified.ml.evaluate import (
    archive_evaluation_report,
    evaluate_prepared,
    prepare_evaluation,
    resolve_checkpoint,
    save_evaluation_report,
)
from overfitted_underqualified.ml.plots import plot_confusion, show_windows

import yaml
import matplotlib.pyplot as plt

evaluation_settings = {**dotenv_values(ROOT / ".env"), **os.environ}
EVAL_PROTOCOL = evaluation_settings.get("EVAL_PROTOCOL") or "annotation_windows"
if EVAL_PROTOCOL not in {"annotation_windows", "full_video", "both"}:
    raise ValueError("EVAL_PROTOCOL: annotation_windows, full_video or both")
RUN_A = EVAL_PROTOCOL in {"annotation_windows", "both"}
RUN_B = EVAL_PROTOCOL in {"full_video", "both"}
print("Протокол:", EVAL_PROTOCOL)


## 1. Checkpoint и сохранённая конфигурация

Показываем доступные `runs/*/best.pt`. Если файл ровно один, выбираем его автоматически.
Если их несколько, укажите `CHECKPOINT_PATH=runs/<нужный запуск>/best.pt` в корневом `.env`
или переменной окружения. Явный неверный путь вызывает ошибку, без незаметной подмены модели.
На локальном компьютере доступны только локальные файлы: скачайте папку запуска с Brev
или выполняйте оценку на Brev. Содержимое данных сверяется с хешами checkpoint.


In [ ]:
# Local .env values can be overridden by environment variables in this kernel.
evaluation_settings = {**dotenv_values(ROOT / ".env"), **os.environ}
available_checkpoints = sorted((ROOT / "runs").glob("*/best.pt"))
display(pd.DataFrame({"available_checkpoint": [str(p.relative_to(ROOT)) for p in available_checkpoints]}))
checkpoint_path = resolve_checkpoint(evaluation_settings.get("CHECKPOINT_PATH"), root=ROOT)
print("Выбран checkpoint:", checkpoint_path)
if RUN_A:
    preparation_start = time.perf_counter()
    evaluation = prepare_evaluation(
        checkpoint_path, device=(evaluation_settings.get("EVAL_DEVICE") or "cpu"),
        events_path=ROOT / "data/annotations/events.csv",
        split_path=ROOT / "data/processed/split_proposal.csv",
        video_dir=Path((evaluation_settings.get("VIDEO_DIR") or str(ROOT / "data/raw/videos"))),
    )
    if evaluation["device"].type == "cuda":
        import torch
        torch.cuda.synchronize(evaluation["device"])
    preparation_sec = time.perf_counter() - preparation_start
    display(evaluation["checkpoint"]["config"])
    print("Checkpoint:", checkpoint_path, "epoch:", evaluation["checkpoint"]["epoch"])
    print("Проверено validation-окон:", len(evaluation["dataset"]))


## 2. Оценка validation

Модель работает в eval mode: параметры и BatchNorm не обновляются. Precision/recall/F1 относятся к violation=1. Accuracy считается по всем окнам. Оценка не перезаписывает результаты обучения.

In [ ]:
if RUN_A:
    import time

    import torch

    from overfitted_underqualified.ml.evaluate import aggregate_video_predictions

    if evaluation["device"].type == "cuda":
        torch.cuda.synchronize(evaluation["device"])
    processing_start = time.perf_counter()
    metrics, rows = evaluate_prepared(evaluation)
    if evaluation["device"].type == "cuda":
        torch.cuda.synchronize(evaluation["device"])
    processing_sec = time.perf_counter() - processing_start
    predictions = pd.DataFrame(rows)
    display(pd.Series({key: value for key, value in metrics.items() if key != "confusion_matrix"}, name="validation"))
    plot_confusion(metrics)
    display(predictions)

    # Distinguish window metrics above from canonical-video metrics below.
    video_metrics, video_rows = aggregate_video_predictions(rows)
    video_predictions = pd.DataFrame(video_rows)
    display(pd.Series({key: value for key, value in video_metrics.items() if key != "confusion_matrix"}, name="video-level validation"))
    plot_confusion(video_metrics)
    display(video_predictions)
    print(f"Оценка окон: {processing_sec:.3f} с; среднее {processing_sec / len(video_rows):.3f} с/видео.")
    print("Включены decode, preprocessing, transfer, inference и оконные метрики; загрузка checkpoint исключена.")
    print("Это время выбранных окон, не полного просмотра видео; warm-up не выполнялся.")


## 3. Просмотр ошибок

Показываем до шести ошибочно классифицированных окон с фиксированным порогом. Это разбор ошибок, а не изменение разметки или настройка порога.

In [ ]:
if RUN_A:
    errors = predictions.loc[predictions["label"].ne(predictions["prediction"])]
    display(errors)
    selected = errors.head(6)
    show_windows(evaluation["dataset"], selected.index,
                 [f"{row.clip_id}; y={row.label}, pred={row.prediction}, p={row.probability:.2f}"
                  for row in selected.itertuples()])
    print("Всего ошибок:", len(errors), "из", len(predictions))


Следующий эксперимент должен проверять конкретную гипотезу. Для вывода о поиске событий нужен отдельный full-video протокол; для переноса на другие камеры/сеансы — независимые данные.

## 4. False positives и false negatives
Разделяем лишние тревоги и пропуски. Для каждого окна доступны исходные ID и время центра.
Показываем обе группы отдельно: пустая группа означает отсутствие таких ошибок на этой выборке,
а не доказательство безошибочности. Кадры — визуальные свидетельства для исследования, не юридическое подтверждение.

In [ ]:
if RUN_A:
    error_analysis = predictions.copy()
    error_analysis["outcome"] = [
        "TP" if truth == pred == 1 else "TN" if truth == pred == 0 else "FP" if pred == 1 else "FN"
        for truth, pred in zip(error_analysis["label"], error_analysis["prediction"])
    ]
    error_analysis["source_ids"] = [list(record.aliases) for record, _ in evaluation["dataset"].samples]
    display(error_analysis.groupby("outcome").agg(windows=("clip_id", "size"), mean_probability=("probability", "mean")))
    for outcome in ("FP", "FN"):
        subset = error_analysis.loc[error_analysis["outcome"].eq(outcome)]
        print(outcome, "окон:", len(subset))
        display(subset)
        selected = subset.head(3)
        show_windows(evaluation["dataset"], selected.index,
                     [f"{outcome}: {row.clip_id}; p={row.probability:.2f}" for row in selected.itertuples()])


## A.5. Временная диагностика: что измеряется сейчас
В протоколе A модель выдаёт вероятность класса окна, **не predicted timestamp**. Поэтому timestamp MAE,
ошибка локализации и доля событий в допуске в A **N/A**, а не ноль.
`center_sec` задан разметкой для положительных окон; сравнивать его с разметкой как предсказание нельзя.
Ниже проверяем только выборку: границы окна, ближайший реально выбранный кадр и его отклонение
от отметки. Это погрешность дискретизации кадров, не ошибка модели. Для FP истинной отметки нет.
Для настоящей оценки локализации нужны full-video inference и заранее заданное сопоставление событий,
включая обработку лишних и пропущенных событий. Контракт FastAPI `Finding.timecode_sec` сохраняется;
он не заполняется центрами известных окон в качестве якобы найденных событий.

In [ ]:
if RUN_A:
    import numpy as np

    from overfitted_underqualified.ml.dataset import sample_times

    timing_rows = []
    dataset = evaluation["dataset"]
    for index, (record, center) in enumerate(dataset.samples):
        requested = sample_times(center, record.last_time, dataset.window_sec, dataset.frames)
        sampled = np.clip(np.rint(requested * record.fps), 0, record.frame_count - 1) / record.fps
        reference = center if record.label == 1 else np.nan
        nearest = sampled[np.argmin(abs(sampled - reference))] if record.label == 1 else np.nan
        timing_rows.append({"clip_id": record.clip_id, "outcome": error_analysis.loc[index, "outcome"],
            "reference_sec": reference, "window_start_sec": sampled.min(), "window_end_sec": sampled.max(),
            "nearest_sample_sec": nearest, "sampling_offset_sec": abs(nearest-reference),
            "predicted_timestamp_sec": np.nan, "timestamp_error_sec": np.nan})
    timing = pd.DataFrame(timing_rows)
    display(timing)
    display(timing["sampling_offset_sec"].describe())
    print("Timestamp localization metrics: N/A — no model timestamp predictions")


## 6. Выводы и следующий исследовательский шаг
Сначала прочитайте итог ниже для выбранного checkpoint. Затем запишите наблюдения по FP/FN:
видно ли действие после crop, хватает ли временного контекста, похожи ли обычные действия на нарушения.
Это гипотезы для следующего эксперимента, а не автоматическая коррекция источника меток.

In [ ]:
if RUN_A:
    counts = error_analysis["outcome"].value_counts()
    print(f"Validation: {len(predictions)} окон; FP={counts.get('FP', 0)}, FN={counts.get('FN', 0)}.")
    print(f"Accuracy={metrics['accuracy']:.3f}, precision={metrics['precision']:.3f}, recall={metrics['recall']:.3f}, F1={metrics['f1']:.3f}.")
    if predictions["prediction"].nunique() == 1:
        print("Модель выдаёт один класс для всех окон. Проверьте дисбаланс предсказаний; recall отдельно недостаточен.")
    print("Локализация не оценена. Независимость камер/сеансов не подтверждена; это не готовый detector.")


Перед коммитом очистите outputs, execution counts и attachments. Не переносите изображения,
описания событий или индивидуальные предсказания в публичные отчёты. Сохранённые runs остаются локальными.

## 7. Автоматическое сохранение и архив для скачивания
Следующая ячейка сохраняет уже рассчитанные результаты без повторного inference.
Каждый запуск создаёт новый каталог `reports/evaluation_<run>_<UTC>_<id>/` и ZIP рядом с ним.
В архив входят метрики, предсказания, FP/FN, временная диагностика, confusion matrix,
provenance и доступные артефакты обучения при совпадении манифеста.
Веса, исходные видео, банковские кадры и `.env` в архив не включаются.
Скачайте ZIP по ссылке в выводе ячейки либо через файловый браузер Jupyter в `reports/`.
После этого архив можно прикрепить в чат для разбора результатов.


In [ ]:
if RUN_A:
    report_name = (
        f"evaluation_{checkpoint_path.parent.name}_"
        f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
    )
    report_dir = ROOT / "reports" / report_name
    saved_report = save_evaluation_report(
        evaluation, checkpoint_path, report_dir, metrics, rows,
        processing_sec=processing_sec, preparation_sec=preparation_sec,
    )
    error_analysis.to_csv(report_dir / "error_analysis.csv", index=False)
    timing.to_csv(report_dir / "sampling_diagnostics.csv", index=False)
    archive_path = archive_evaluation_report(report_dir, checkpoint_path)
    print("Результаты:", report_dir)
    print("Архив для отправки:", archive_path)
    display(FileLink(os.path.relpath(archive_path, Path.cwd()), result_html_prefix="Скачать ZIP: "))


## B.1. Полное видео: фиксированный протокол
Выбор выше: EVAL_PROTOCOL=full_video или both. Для B настройки читаются до предсказаний из configs/full_video.yaml.
Порог 0.5, шаг 2 с, допуск сопоставления 5 с — исследовательские значения, не подобранные на validation.
Оцениваем все 10 канонических validation-видео. Разные отметки одного видео остаются отдельными событиями.
Не загружаем pretrained-веса и не обновляем BatchNorm. Подготовка сверяет checkpoint, SHA-256, split и метки.


In [ ]:
from overfitted_underqualified.ml.full_video_evaluation import (
    prepare_full_evaluation, evaluate_full_prepared, save_full_report, archive_full_report,
)
from overfitted_underqualified.ml.inference import ScanConfig, evidence_frames
from overfitted_underqualified.ml.plots import show_evidence

if RUN_B:
    full_config = yaml.safe_load((ROOT / "configs/full_video.yaml").read_text(encoding="utf-8"))
    display(full_config)
    full_prepared = prepare_full_evaluation(
        checkpoint_path, ScanConfig(**full_config["scan"]),
        device=evaluation_settings.get("EVAL_DEVICE") or "cpu",
        events_path=ROOT / "data/annotations/events.csv", split_path=ROOT / "data/processed/split_proposal.csv",
        video_dir=Path(evaluation_settings.get("VIDEO_DIR") or str(ROOT / "data/raw/videos")),
    )
    display(pd.DataFrame([{"clip_id": r.clip_id, "label": r.label, "sha256": r.sha256}
                          for r in full_prepared["records"]]))


In [ ]:
if RUN_B:
    full_result = evaluate_full_prepared(full_prepared, tolerance_sec=full_config["matching_tolerance_sec"])
    full_summary = full_result["summary"]
    full_videos = pd.DataFrame(full_summary["videos"])
    display(full_videos)
    display(pd.Series(full_summary["metrics"]["video"], name="B: full-video"))
    plot_confusion(full_summary["metrics"]["video"], threshold=full_config["scan"]["threshold"])
    display(pd.Series(full_summary["metrics"]["false_alarms"], name="Тревоги на отрицательных видео"))
    print("Полный проход, секунд:", full_result["processing_sec"])
    if RUN_A:
        display(pd.DataFrame([
            {"protocol": "A: annotation-centred canonical video aggregation", **video_metrics},
            {"protocol": "B: full-video scan", **full_summary["metrics"]["video"]},
        ]))
        print("Разный охват окон. Разность F1 не является измерением улучшения одной и той же задачи.")


## B.2. Временное сопоставление и пропуски
Алгоритм максимизирует число пар в заданном допуске, затем минимизирует суммарную абсолютную ошибку.
Каждая отметка и кандидат используются максимум один раз. Лишний кандидат возле уже сопоставленной отметки
остаётся несопоставленным. Для положительных видео это unresolved, а не подтверждённый FP: разметка неполна.
MAE относится только к matched-парам — читайте его вместе с числом пропущенных отметок.


In [ ]:
if RUN_B:
    temporal_matches = pd.DataFrame(full_summary["matching"])
    display(pd.Series(full_summary["metrics"]["temporal"], name="Предварительно относительно известных отметок"))
    display(temporal_matches)
    display(temporal_matches.loc[temporal_matches["status"].ne("matched")])
    matched = temporal_matches.loc[temporal_matches["status"].eq("matched")]
    if not matched.empty:
        matched.plot.bar(x="clip_id", y="signed_error_sec", ylabel="Ошибка предполагаемого времени, с")
        plt.show()
        plt.close("all")


## B.3. Ошибки и локальные кадры
Сначала рассмотрите video-level FP/FN. В выбранном видео видны вероятность, поддержки кандидатов и известные
отметки (они добавлены только на график после inference). Для пропуска без кандидатов показывается контекст
исходной отметки, явно как reference. Это не предсказанное время. Кадры не сохраняются в отчёт/ZIP.


In [ ]:
if RUN_B:
    full_errors = full_videos.loc[full_videos["outcome"].isin(["FP", "FN"])]
    display(full_errors)
    SELECTED_VIDEO = full_errors.iloc[0]["clip_id"] if not full_errors.empty else full_videos.iloc[0]["clip_id"]
    record = next(r for r in full_prepared["records"] if r.clip_id == SELECTED_VIDEO)
    scan = next(s for s in full_result["scans"] if s["clip_id"] == SELECTED_VIDEO)
    trace = pd.DataFrame(scan["windows"])
    ax = trace.plot(x="center_sec", y="probability", figsize=(12, 3), ylim=(0, 1), marker=".")
    ax.axhline(full_config["scan"]["threshold"], color="red", linestyle="--")
    for c in scan["candidates"]:
        ax.axvspan(c["start_sec"], c["end_sec"], color="orange", alpha=0.2)
        ax.axvline(c["timestamp_sec"], color="orange")
    for reference in record.event_times:
        ax.axvline(reference, color="green", linestyle=":")
    ax.set_title(f"{SELECTED_VIDEO}: orange=predicted candidate; green=known reference")
    plt.show()
    plt.close("all")
    display(pd.DataFrame(scan["candidates"]))
    for candidate in scan["candidates"][:3]:
        show_evidence(evidence_frames(record.path, candidate["timestamp_sec"]),
                      f"{SELECTED_VIDEO}: контекст кандидата {candidate['candidate_id']}, тип неизвестен")
    if not scan["candidates"] and record.event_times:
        show_evidence(evidence_frames(record.path, min(record.event_times[0], record.last_time)),
                      f"{SELECTED_VIDEO}: REFERENCE пропущенной отметки, не предсказание")


## B.4. Вывод и архив
Есть бинарный поиск кандидатов с предполагаемыми временами. Квалификация действия и юридического нарушения
не выполнена. Общая комната, люди и неизвестные сеансы ограничивают переносимость оценки; checkpoint ранее
выбирался по этой validation. Наблюдения по кадрам — гипотезы, не изменения меток.
Следующая ячейка автоматически сохраняет уже полученные результаты и ZIP по allowlist: без банковских кадров,
видео, весов и .env. Для отдельного сравнения запусков сохраняйте разные протоколы и конфигурации.


In [ ]:
if RUN_B:
    full_report_dir = ROOT / "reports" / f"evaluation_full_{checkpoint_path.parent.name}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
    full_report = save_full_report(full_prepared, full_result, full_report_dir)
    full_archive = archive_full_report(full_report_dir)
    temporal = full_report["metrics"]["temporal"]
    print(f"Совпало {temporal['matched']} из {temporal['annotations']} известных отметок; пропущено {temporal['unmatched_annotations']}.")
    print("Несопоставленных кандидатов:", temporal["unmatched_predictions"], "; MAE только matched:", temporal["matched_mae_sec"])
    print("Отчёт:", full_report_dir)
    display(FileLink(os.path.relpath(full_archive, Path.cwd()), result_html_prefix="Скачать полный проход ZIP: "))
